# 01 · Construcción de pools y control de calidad

Este notebook cubre la **sección 2** de la práctica:

1. Descargar (o cargar desde local) el archivo oficial de la TLC (enero 2015).
2. Definir el depósito y los 10 sitios que serán nuestros nodos `V`.
3. Aplicar filtros sobre los registros (ver sección 2.3).
4. Construir los pools $P_{ij}$ por cada arco.
5. Revisar calidad: deben aparecer $|A|=110$ pools, y el más pequeño debe tener al menos 87 registros.
6. Calcular la distancia determinista $d^{road}_{ij}$ y el costo de primera etapa $c_{ij} = \kappa \, d^{road}_{ij}$.

**Archivos generados:**
- `data/processed/pools.parquet` → una fila por viaje válido (`i`, `j`).
- `data/processed/arc_costs.csv` → una fila por arco, con $d^{road}_{ij}$ y $c_{ij}$.

## 1. Configuración inicial de rutas y archivos

En esta celda se importan las librerías necesarias y se definen las carpetas de trabajo.  
- `RAW_DIR` apunta a los datos crudos.  
- `PROC_DIR` es la carpeta donde se guardarán los datos procesados (se crea si no existe).  
- `RAW_FILE_JAN` corresponde al archivo de viajes de enero 2015 en formato Parquet.  

Con esto dejamos listo el entorno para empezar a trabajar con los datos.

In [6]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE_JAN = RAW_DIR / "yellow_tripdata_2015-01.parquet"

## 2. Nodos y zonas TLC

Depósito + 10 sitios, según la tabla de la sección 2.2.

In [7]:
sites = pd.DataFrame([
    {"i": 0,  "name": "Javits Center (Depósito)",     "lat": 40.75750, "lon": -74.00250, "location_id": 246},
    {"i": 1,  "name": "Times Square",                 "lat": 40.75800, "lon": -73.98550, "location_id": 230},
    {"i": 2,  "name": "Rockefeller Center",            "lat": 40.75870, "lon": -73.97870, "location_id": 161},
    {"i": 3,  "name": "Grand Central Terminal",        "lat": 40.75278, "lon": -73.97722, "location_id": 162},
    {"i": 4,  "name": "New York Public Library",       "lat": 40.75306, "lon": -73.98194, "location_id": 164},
    {"i": 5,  "name": "Union Square",                  "lat": 40.73590, "lon": -73.99110, "location_id": 234},
    {"i": 6,  "name": "Washington Square Park",        "lat": 40.73083, "lon": -73.99750, "location_id": 114},
    {"i": 7,  "name": "Madison Square Garden",         "lat": 40.75056, "lon": -73.99361, "location_id": 186},
    {"i": 8,  "name": "One World Trade Center",        "lat": 40.71274, "lon": -74.01338, "location_id": 261},
    {"i": 9,  "name": "New York Stock Exchange",       "lat": 40.70693, "lon": -74.01125, "location_id": 87},
    {"i": 10, "name": "South Street Seaport (Pier 17)","lat": 40.70600, "lon": -74.00270, "location_id": 209},
])

location_to_i = dict(zip(sites["location_id"], sites["i"]))
sites


,i,name,lat,lon,location_id
0,0,Javits Center (Depósito),40.75750,-74.00250,246
1,1,Times Square,40.75800,-73.98550,230
2,2,Rockefeller Center,40.75870,-73.97870,161
3,3,Grand Central Terminal,40.75278,-73.97722,162
4,4,New York Public Library,40.75306,-73.98194,164
5,5,Union Square,40.73590,-73.99110,234
6,6,Washington Square Park,40.73083,-73.99750,114
7,7,Madison Square Garden,40.75056,-73.99361,186
8,8,One World Trade Center,40.71274,-74.01338,261
9,9,New York Stock Exchange,40.70693,-74.01125,87


## 3. Filtros y construcción de pools 

En esta celda aplicamos directamente los filtros en DuckDB sobre el archivo Parquet, proyectando solo las columnas que nos interesan.  
Los criterios son:

1. Viajes con pickup entre el 1 y el 31 de enero de 2015, de lunes a viernes, en horario local de `[09:00, 17:00)`.
2. Origen y destino en zonas distintas según la tabla de sitios.
3. Duración `t` (en minutos) entre 1 y 90.
4. `trip_distance` entre 0.1 y 30 millas.

In [4]:
location_ids = tuple(sites["location_id"].tolist())

query = f"""
SELECT
    PULocationID,
    DOLocationID,
    trip_distance,
    date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime) / 60.0 AS duration_min,
    dayofweek(tpep_pickup_datetime) AS dow,   -- 0=domingo ... 6=sábado en DuckDB
    hour(tpep_pickup_datetime) AS pickup_hour
FROM read_parquet('{RAW_FILE_JAN.as_posix()}')
WHERE tpep_pickup_datetime >= TIMESTAMP '2015-01-01 00:00:00'
  AND tpep_pickup_datetime <  TIMESTAMP '2015-02-01 00:00:00'
  AND PULocationID IN {location_ids}
  AND DOLocationID IN {location_ids}
  AND PULocationID != DOLocationID
"""

raw = duckdb.sql(query).df()
print(f"Registros tras filtro de zonas/fecha bruta: {len(raw):,}")
raw.head()


Registros tras filtro de zonas/fecha bruta: 733,944


,PULocationID,DOLocationID,trip_distance,duration_min,dow,pickup_hour
0,234,161,2.60,32.233333,4,0
1,230,161,1.65,14.366667,4,0
2,234,186,0.87,7.616667,4,0
3,186,234,1.46,9.400000,4,0
4,161,162,0.50,4.583333,4,0


In [ ]:
# Filtro de día de semana (lunes-viernes) y franja horaria [09:00, 17:00)
mask_weekday = raw["dow"].between(1, 5)
mask_hour = raw["pickup_hour"].between(9, 16)  
mask_duration = raw["duration_min"].between(1, 90)
mask_distance = raw["trip_distance"].between(0.1, 30)

filtered = raw[mask_weekday & mask_hour & mask_duration & mask_distance].copy()
print(f"Registros tras todos los filtros: {len(filtered):,}")

filtered["i"] = filtered["PULocationID"].map(location_to_i)
filtered["j"] = filtered["DOLocationID"].map(location_to_i)
filtered = filtered.dropna(subset=["i", "j"]).astype({"i": int, "j": int})
filtered.head()


Registros tras todos los filtros: 242,841


,PULocationID,DOLocationID,trip_distance,duration_min,dow,pickup_hour,i,j
2979,261,186,3.10,9.250000,4,9,8,7
2980,186,164,0.60,3.783333,4,9,7,4
2981,230,161,0.44,4.166667,4,9,1,2
2983,164,261,3.00,11.000000,4,9,4,8
2984,186,246,0.80,4.566667,4,9,7,0


## 4. Control de calidad

En este paso verificamos que se cumpla lo esperado:

- Deben aparecer $|A| = 10 \times 11 = 110$ pares dirigidos con al menos un registro.  
- El pool más pequeño debe tener al menos 87 registros.  

Si los números no coinciden, conviene revisar primero la franja horaria, las unidades y el filtro de fecha antes de continuar.

In [ ]:
pool_counts = (
    filtered.groupby(["i", "j"])
    .size()
    .reset_index(name="n_trips")
    .sort_values("n_trips")
)

n_arcs_total = 10 * 11  
n_arcs_present = len(pool_counts)

print(f"Arcos con al menos 1 registro: {n_arcs_present} / {n_arcs_total}")
print(f"Tamaño mínimo de pool: {pool_counts['n_trips'].min()}")
print(f"Tamaño máximo de pool: {pool_counts['n_trips'].max()}")

assert n_arcs_present == 110, "No aparecen los 110 pares dirigidos esperados."
assert pool_counts["n_trips"].min() >= 87, "Algún pool tiene menos de 87 registros; revisar filtros."

pool_counts.head(10)


Arcos con al menos 1 registro: 110 / 110
Tamaño mínimo de pool: 87
Tamaño máximo de pool: 8826


,i,j,n_trips
106,10,6,87
9,0,10,89
100,10,0,101
69,6,10,115
104,10,4,120
89,8,10,121
19,1,10,124
108,10,8,124
101,10,1,133
107,10,7,136


## 5. Distancia determinista y costo de primera etapa

$$d^{road}_{ij} = 1.60934 \times \text{mediana}\{trip\_distance_r : r \in R_{ij}\} \quad [km]$$
$$c_{ij} = \kappa \, d^{road}_{ij}, \quad \kappa = 1 \text{ dólar/km}$$

In [7]:
KAPPA = 1.00  # dólares/km
MILES_TO_KM = 1.60934

arc_costs = (
    filtered.groupby(["i", "j"])["trip_distance"]
    .median()
    .reset_index(name="median_miles")
)
arc_costs["d_road_km"] = arc_costs["median_miles"] * MILES_TO_KM
arc_costs["c_ij"] = KAPPA * arc_costs["d_road_km"]
arc_costs = arc_costs.merge(pool_counts, on=["i", "j"])

arc_costs.sort_values(["i", "j"]).head(10)


,i,j,median_miles,d_road_km,c_ij,n_trips
0,0,1,1.40,2.253076,2.253076,3228
1,0,2,1.80,2.896812,2.896812,2661
2,0,3,2.20,3.540548,3.540548,1322
3,0,4,1.37,2.204796,2.204796,2165
4,0,5,1.39,2.236983,2.236983,3440
5,0,6,2.30,3.701482,3.701482,501
6,0,7,0.90,1.448406,1.448406,3118
7,0,8,3.70,5.954558,5.954558,297
8,0,9,4.30,6.920162,6.920162,415
9,0,10,3.72,5.986745,5.986745,89


## 6. Guardar resultados

- `pools.parquet`: un registro por viaje válido (para muestrear escenarios en el notebook 02).
- `arc_costs.csv`: $d^{road}_{ij}$, $c_{ij}$ y tamaño de pool por arco (para la formulación del modelo).

In [8]:
pools_out = filtered[["i", "j", "duration_min"]].rename(columns={"duration_min": "t"})
pools_out.to_parquet(PROC_DIR / "pools.parquet", index=False)

arc_costs.to_csv(PROC_DIR / "arc_costs.csv", index=False)
sites.to_csv(PROC_DIR / "sites.csv", index=False)

print("Guardado:")
print(f"  {PROC_DIR / 'pools.parquet'}  ({len(pools_out):,} filas)")
print(f"  {PROC_DIR / 'arc_costs.csv'}  ({len(arc_costs)} arcos)")
print(f"  {PROC_DIR / 'sites.csv'}")


Guardado:
  ../data/processed/pools.parquet  (242,841 filas)
  ../data/processed/arc_costs.csv  (110 arcos)
  ../data/processed/sites.csv
